<a href="https://colab.research.google.com/github/Subhranshu-123/BIKE2/blob/main/QUESTION_ANSWERING_NLP_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports the core PyTorch library.

Imports the high-performance, Rust-backed BERT Tokenizer from Hugging Face.

Imports a dedicated BERT model architecture wrapper configured for extractive question answering.

Imports the AdamW optimizer class.

In [ ]:
import torch
from transformers import BertTokenizerFast, BertForQuestionAnswering
from torch.optim import AdamW

• Loads a fast tokenizer to convert raw questions and text passages into numerical sub-word tokens that the model understands.

• Loads the base BERT model along with a specialized question-answering linear output layer placed on top.

• Downloads pre-trained weights (bert-base-uncased) from Hugging Face that were trained on massive text data in lowercase format.

• Prepares both components to instantly start either extractive answer training or inference workloads.

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
model = BertForQuestionAnswering.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

GIVE SOME CONTEXT WITH ANSWER_TEXT AS WELL

In [ ]:
context = "BERT is a transformers-based machine learning technique developed by Google."
question = "Who developed BERT?"
answer_text = "Google"

• Locates the answer start index by finding the exact character position where the answer text begins inside the context string.

• Calculates the answer end index by adding the length of the answer string to its starting position.

• Establishes raw boundaries in character space, which are needed before converting them into token-level indices for BERT.

• Assumes exact text matching, meaning the answer phrase must exist in the context paragraph exactly as written.

In [ ]:
start_char_idx = context.find(answer_text)
end_char_idx = start_char_idx + len(answer_text)

• Tokenizes and formats inputs by pairing the question and context strings together into numerical token IDs with structural separator tags.

• Limits length and truncates text to a maximum sequence boundary of 512 tokens, cutting down only the context section if the combined text is too long.

• Tracks raw character positions by enabling return_offsets_mapping=True, which creates a map linking each generated token back to its character index in the original text.

• Outputs PyTorch tensors (return_tensors="pt") so the data structure is instantly ready to be passed directly into the neural network layers.

In [ ]:
inputs = tokenizer(
    question,
    context,
    max_length=512,
    truncation="only_second", # Only truncate the context, keep the question
    return_offsets_mapping=True,
    return_tensors="pt"
)

• Extracts and separates character mappings by taking the offset coordinates out of the input dictionary and isolating the first batch sequence.

• Retrieves sequence segment identifiers (sequence_ids()) to distinguish which individual tokens belong to the question versus the context passage.

• Identifies special structural tokens by mapping non-text elements (like [CLS] and [SEP]) to None, the question tokens to 0, and context tokens to 1.

• Prepares the positioning logic needed to safely translate raw character locations into precise token search boundaries for the model loss function.

In [ ]:
offset_mapping = inputs.pop("offset_mapping")[0]

sequence_ids = inputs.sequence_ids()

• Locates the context text start by scanning forward from the beginning of the sequence until it hits the first token belonging to the context (1).

• Finds the context text end by scanning backward from the very last token until it encounters the final token belonging to the context.

• Establishes search boundaries (context_start and context_end) that completely isolate the passage tokens while ignoring the question and special structural tags.

• Prevents out-of-bounds scanning errors by pinpointing the exact range where the model is allowed to search for and extract the answer span.

In [ ]:
idx = 0
while sequence_ids[idx] != 1:
    idx += 1
context_start = idx

idx = len(sequence_ids) - 1
while sequence_ids[idx] != 1:
    idx -= 1
context_end = idx

• Checks for answer truncation by determining if the ground-truth answer lies completely outside the context text boundaries that were cut off during tokenization.

• Resets to the classification token (0) if the answer is missing or clipped, pointing the labels directly at the [CLS] token position.

• Scans context tokens forward to match the character start index and pin down the exact starting token position of the answer.

• Scans context tokens backward from the end of the text passage to lock in the precise ending token position of the answer.

• Calculates exact model labels (start_position and end_position) required to calculate classification loss during the training process.

In [ ]:
if offset_mapping[context_start][0] > start_char_idx or offset_mapping[context_end][1] < end_char_idx:
    start_position = 0
    end_position = 0
else:
    # Update token positions
    start_position = context_start
    while start_position <= context_end and offset_mapping[start_position][0] <= start_char_idx:
        start_position += 1
    start_position -= 1

    end_position = context_end
    while end_position >= context_start and offset_mapping[end_position][1] >= end_char_idx:
        end_position -= 1
    end_position += 1

• Appends the target answers directly into the input dictionary as new key-value pairs representing the ground-truth locations.

• Wraps indices into PyTorch tensors (torch.tensor) to match the numerical type required by the model's loss calculation layers.

• Prepares the payload for training so that passing *inputs into BERT automatically calculates and returns the Cross-Entropy loss.

In [ ]:
inputs["start_positions"] = torch.tensor([start_position])
inputs["end_positions"] = torch.tensor([end_position])


In [ ]:
optimizer = AdamW(model.parameters(), lr=3e-5)
model.train()

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elem

• Initializes the AdamW optimizer with a standard learning rate o to manage updates for all of the BERT model's trainable parameters.

• Sets the model to training mode (model.train()), activating vital training behaviors like dropout layers to prevent overfitting.

• Prepares the entire system to start processing forward passes and executing backward gradient updates safely.

In [ ]:
optimizer.zero_grad()
outputs = model(**inputs)

• Extracts the training loss value calculated automatically by the model's internal Cross-Entropy loss layer based on the predicted answer span vs. the ground truth.

• Triggers the backward pass (loss.backward()) to compute gradients for every trainable parameter in the network graph using automatic differentiation.

• Updates the network weights (optimizer.step()) by applying the calculated gradients through the AdamW optimization algorithm to complete the training step.

In [ ]:
loss = outputs.loss
loss.backward()
optimizer.step()

print the result

In [ ]:
print(f"Training Step Complete. Loss: {loss.item():.4f}")

Training Step Complete. Loss: 3.3638


In [ ]:
import torch
from transformers import BertTokenizerFast, BertForQuestionAnswering

def get_answer(question: str, context: str, model_path: str = "bert-base-uncased") -> str:
    """
    Extracts an answer span from a context paragraph using a fine-tuned BERT model.
    """
    # 1. Load fast tokenizer and model (Ensure model is in evaluation mode)
    tokenizer = BertTokenizerFast.from_pretrained(model_path)
    model = BertForQuestionAnswering.from_pretrained(model_path)
    model.eval()

    # 2. Tokenize the input text pairs
    inputs = tokenizer(
        question,
        context,
        max_length=512,
        truncation="only_second",
        return_tensors="pt"
    )

    # 3. Model inference (disable gradient tracking)
    with torch.no_grad():
        outputs = model(**inputs)

    # 4. Extract logits for start and end positions
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    # Get the highest-probability token indices
    start_idx = torch.argmax(start_logits)
    end_idx = torch.argmax(end_logits)

    # 5. Convert token IDs back to a readable string string
    all_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # If the end index is logically before the start index, extraction failed
    if end_idx < start_idx:
        return "No clear answer found."

    # Slice the token array to fetch the text fragment
    answer_tokens = all_tokens[start_idx : end_idx + 1]

    # Clean up WordPiece artifacts (e.g., ["Google", "'", "s"] or ["trans", "##formers"])
    answer = tokenizer.convert_tokens_to_string(answer_tokens)

    return answer


context_paragraph = "The Eiffel Tower is located in Paris, France. It was constructed from 1887 to 1889."
user_query = "When was the Eiffel Tower built?"

extracted_answer = get_answer(question=user_query, context=context_paragraph)
print(f"Question: {user_query}")
print(f"Answer:   {extracted_answer}")



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Question: When was the Eiffel Tower built?
Answer:   ##iffel tower


In [ ]:
!apt-get install -y tesseract-ocr
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [ ]:


import torch
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from PIL import Image

# 1. Load the Pre-trained Processor and Model
# 'layoutlmv3-base' can be fine-tuned for token classification (NER on documents)
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=True)
model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", num_labels=5)
model.eval()

# 2. Load Your Document Image
# Make sure you have 'pytesseract' installed so the processor can run OCR automatically
image_path = "/content/invoice.jpeg"
image = Image.open(image_path).convert("RGB")

# 3. Process the Image
# The processor automatically extracts text, scales the layout bounding boxes,
# and crops visual image patches.
encoding = processor(
    image,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# 4. Model Inference
with torch.no_grad():
    outputs = model(**encoding)

# 5. Extract Predictions
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
tokens = processor.tokenizer.convert_ids_to_tokens(encoding["input_ids"].squeeze().tolist())

# Print out tokens matched with predicted class IDs
for token, pred in zip(tokens, predictions):
    if token not in ['[CLS]', '[SEP]', '[PAD]']:
        print(f"Token: {token:<15} -> Predicted Class ID: {pred}")

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Token: <s>             -> Predicted Class ID: 4
Token: ĠIN             -> Predicted Class ID: 4
Token: VO              -> Predicted Class ID: 0
Token: ICE             -> Predicted Class ID: 0
Token: ĠApex           -> Predicted Class ID: 4
Token: ĠConsulting     -> Predicted Class ID: 4
Token: ĠGroup          -> Predicted Class ID: 4
Token: Ġ123            -> Predicted Class ID: 4
Token: ĠBusiness       -> Predicted Class ID: 4
Token: ĠAve            -> Predicted Class ID: 4
Token: ,               -> Predicted Class ID: 0
Token: ĠSuite          -> Predicted Class ID: 4
Token: Ġ400            -> Predicted Class ID: 4
Token: ĠMet            -> Predicted Class ID: 4
Token: ropolis         -> Predicted Class ID: 4
Token: ,               -> Predicted Class ID: 0
Token: ĠNY             -> Predicted Class ID: 4
Token: Ġ100            -> Predicted Class ID: 4
Token: 01              -> Predicted Class ID: 0
Token: ĠA              -> Predicted Class ID: 4
Token: ĠBill           -> Predicted Clas

In [ ]:
import re
import string
from collections import Counter

def normalize_text(text: str) -> str:
    """
    Applies standard SQuAD text normalization:
    - Converts text to lowercase
    - Removes punctuation
    - Strips articles (a, an, the)
    - Fixes white spaces
    """
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(text))))


def compute_exact_match(prediction: str, truth: str) -> int:
    """Returns 1 if the prediction matches the truth exactly, else 0."""
    return int(normalize_text(prediction) == normalize_text(truth))


def compute_f1_score(prediction: str, truth: str) -> float:
    """Computes token-level F1-score between prediction and truth strings."""
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(truth).split()

    # If either sequence is empty, F1 is 1 if both are empty, else 0
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return float(pred_tokens == truth_tokens)

    # Count token frequencies
    common_tokens = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common_tokens.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)

    f1 = (2 * precision * recall) / (precision + recall)
    return f1


#  Evaluation with the Invoice Generated Image Data
# Ground Truth Answer from the previous invoice: "$9,315.00"
ground_truth = "$9,315.00"

# Scenario A: Model predicts text with minor variation (e.g., missed the dollar sign)
prediction_a = "9,315.00"

# Scenario B: Model predicts a completely wrong string
prediction_b = "Cloud Infrastructure Setup"

print("Evaluation Scenario A ")
print(f"Ground Truth: '{ground_truth}' | Prediction: '{prediction_a}'")
print(f"Exact Match (EM): {compute_exact_match(prediction_a, ground_truth)}")
print(f"F1-Score:         {compute_f1_score(prediction_a, ground_truth):.4f}\n")

print("Evaluation Scenario B")
print(f"Ground Truth: '{ground_truth}' | Prediction: '{prediction_b}'")
print(f"Exact Match (EM): {compute_exact_match(prediction_b, ground_truth)}")
print(f"F1-Score:         {compute_f1_score(prediction_b, ground_truth):.4f}")


Evaluation Scenario A 
Ground Truth: '$9,315.00' | Prediction: '9,315.00'
Exact Match (EM): 1
F1-Score:         1.0000

Evaluation Scenario B
Ground Truth: '$9,315.00' | Prediction: 'Cloud Infrastructure Setup'
Exact Match (EM): 0
F1-Score:         0.0000
